In [ ]:
!pip install huggingface_hub onnxruntime-genai

In [ ]:
import os
import json
import shutil
import onnxruntime_genai as og
from huggingface_hub import snapshot_download

LOCAL_DIR = "./local_llama3_onnx"
os.makedirs(LOCAL_DIR, exist_ok=True)

# 1. Download unblocked structures from the open community repository
REMOTE_REPO = "onnx-community/Llama-3.2-1B-Instruct-GENAI-ONNX"
print(f"📦 Zscaler-Safe Channel: Fetching runtime files from {REMOTE_REPO}...")
snapshot_download(
    repo_id=REMOTE_REPO,
    local_dir=LOCAL_DIR,
    ignore_patterns=["cuda/*", "directml/*", "webgpu/*"] # Target CPU/Mobile layers
)

# 2. Subfolder Scan and Auto-Flatten Layer
print("🔍 Scanning subfolders for hidden ONNX model binaries...")
onnx_found = False
for root, dirs, files in os.walk(LOCAL_DIR):
    # Skip moving if we are already in the target root folder
    if os.path.abspath(root) == os.path.abspath(LOCAL_DIR):
        continue
    for file in files:
        # Move all core modeling components up to the main root folder
        if file.endswith(".onnx") or file.endswith(".onnx.data") or "token" in file or file == "config.json":
            src_path = os.path.join(root, file)
            dest_path = os.path.join(LOCAL_DIR, file)
            shutil.move(src_path, dest_path)
            if file.endswith(".onnx"):
                onnx_found = True

# Detect if the repository uses a custom file name or standard naming convention
model_filename = "model.onnx"
if onnx_found:
    onnx_files = [f for f in os.listdir(LOCAL_DIR) if f.endswith(".onnx")]
    if onnx_files:
        model_filename = onnx_files[0]
        print(f"✅ Found and moved model binary: {model_filename}")

# 3. Generate a corrected genai_config.json manifest pointing to the exact binary name
genai_config_data = {
    "model": {
        "bos_token_id": 128000,
        "context_length": 131072,
        "decoder": {
            "session_options": {
                "log_id": "onnxruntime-genai"
            },
            "filename": model_filename, # Dynamic assignment ensuring accurate linking
            "head_size": 64,
            "hidden_size": 2048,
            "inputs": {
                "input_ids": "input_ids",
                "attention_mask": "attention_mask",
                "position_ids": "position_ids",
                "past_key_names": "past_key_values.%d.key",
                "past_value_names": "past_key_values.%d.value"
            },
            "outputs": {
                "logits": "logits",
                "present_key_names": "present.%d.key",
                "present_value_names": "present.%d.value"
            },
            "num_attention_heads": 32,
            "num_hidden_layers": 16,
            "num_key_value_heads": 8
        },
        "eos_token_id": [128001, 128004, 128008, 128009],
        "pad_token_id": 128001,
        "type": "llama",
        "vocab_size": 128256
    },
    "search": {
        "diversity_penalty": 0.0,
        "do_sample": False,
        "early_stopping": True,
        "length_penalty": 1.0,
        "max_length": 2048,
        "min_length": 0,
        "no_repeat_ngram_size": 0,
        "num_beams": 1,
        "num_return_sequences": 1,
        "past_present_share_buffer": False,
        "repetition_penalty": 1.0,
        "temperature": 0.1,
        "top_k": 1,
        "top_p": 0.9
    }
}

with open(os.path.join(LOCAL_DIR, "genai_config.json"), "w", encoding="utf-8") as f:
    json.dump(genai_config_data, f, indent=4)
print("📝 Structural configuration layout updated successfully.")

# 4. Load the compiled model directory directly into the execution engine
print("\n🤖 Booting local model engine...")
model = og.Model(LOCAL_DIR)
tokenizer = og.Tokenizer(model)
tokenizer_stream = tokenizer.create_stream()


In [ ]:
# Construct your prompt using the correct ChatML format
prompt = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "You are an expert Python code reviewer. Identify critical runtime bugs, styling issues, and suggest robust corrections.<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>\n\n"
    "Perform a concise code review on this function:\n```python\ndef process(data):\n    x = data/0\n    return x\n```<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
)

# FIXED: Separate token encoding and generator generation steps correctly
input_tokens = tokenizer.encode(prompt)

params = og.GeneratorParams(model)
params.set_search_options(temperature=0.1, max_length=512)

print("\n=== Llama-3.2's High-Quality Code Review ===")
# FIXED: Passing tokens directly to the Generator constructor alongside configurations
generator = og.Generator(model, params)
generator.append_tokens(input_tokens)

while not generator.is_done():
    generator.generate_next_token()
    new_token = generator.get_sequence(0)[-1]
    print(tokenizer_stream.decode(new_token), end="", flush=True)
print("\n")




In [ ]:
!pip install transformers torch huggingface_hub accelerate


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Using the official Meta repository which natively stores models in safetensors format
MODEL_REPO = "meta-llama/Llama-3.2-1B-Instruct"
LOCAL_DIR = "./local_llama3_safetensors"

# 1. Download/Load tokenizer and model weights securely
print(f"📦 Zscaler-Safe Channel: Fetching Safetensors weights from {MODEL_REPO}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, cache_dir=LOCAL_DIR)

# Load the model directly onto the CPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    cache_dir=LOCAL_DIR,
    torch_dtype=torch.float32, # Standard precision for CPU architecture
    device_map="cpu"           # Forces calculation completely on your CPU
)

# 2. Build a text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)



In [ ]:
# 3. Your Original Chat Layout Payload
chat_messages = [
    {"role": "system", "content": "You are an expert Python code reviewer. Identify critical runtime bugs, styling issues, and suggest robust corrections."},
    {"role": "user", "content": f"Perform a concise code review on this function:\n```python\ndef process(data):\n    x = data/0\n    return x\n```"}
]

# Format the message payload exactly into the Llama-3 text template structure
prompt = pipe.tokenizer.apply_chat_template(
    chat_messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\n🤖 Running strict instruction-based code review on CPU...")
# 4. Generate the response matching your temperature and token limits
outputs = pipe(
    prompt,
    max_new_tokens=512,
    do_sample=False, # Setting to False guarantees exact debugging determinism
    temperature=0.1,
    eos_token_id=pipe.tokenizer.eos_token_id,
    pad_token_id=pipe.tokenizer.eos_token_id
)

print("\n=== Llama-3.2 Safetensors High-Quality Code Review ===")
# Clean up and print the generation output text
generated_text = outputs[0]["generated_text"]
response_content = generated_text[len(prompt):]
print(response_content.strip())
